In [0]:
import json
from pathlib import Path
import pandas as pd
from pyspark.sql.functions import *

In [0]:

dbutils.widgets.text("raw_root", "/Volumes/workspace/raw/proyecto/openweathermap", "Raw Root")
# Léalos en Python
raw_root = Path(dbutils.widgets.get("raw_root"))

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.bronze
COMMENT 'Capa Bronze: datos crudos procesados'

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.bronze.hist_air_pollution")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.bronze.hist_air_pollution (
    dt STRING,  -- timestamp UNIX en segundos
    main STRING,  -- guardamos el struct como JSON string
    components STRING,  -- guardamos el struct como JSON string
    ubigeo STRING,
    distrito STRING,
    provincia STRING,
    departamento STRING,
    latitud STRING,
    longitud STRING,
    fecha STRING, -- fecha derivada de dt
    fecha_actualizacion STRING -- fecha en que se actualiza la tabla
) USING DELTA
PARTITIONED BY (fecha);

In [0]:
candidate_files = sorted(raw_root.glob("**/*.json"))

records = []
for json_file in candidate_files:
    # cada archivo es JSON Lines
    df = pd.read_json(json_file, lines=True, encoding="utf-8")

    # aseguramos que main y components se serialicen como JSON string
    if "main" in df.columns:
        df["main"] = df["main"].apply(lambda x: json.dumps(x) if isinstance(x, dict) else str(x))
    if "components" in df.columns:
        df["components"] = df["components"].apply(lambda x: json.dumps(x) if isinstance(x, dict) else str(x))

    records.extend(df.to_dict(orient="records"))

# convertir a Spark DataFrame
df_bronze = spark.createDataFrame(pd.DataFrame(records))
df_bronze = df_bronze.withColumn("fecha", to_date(to_timestamp(col("dt").cast("long")))).withColumn("fecha_actualizacion", lit(current_date()))

# convertir todas las columnas a STRING
for col_name in df_bronze.columns:
    df_bronze = df_bronze.withColumn(col_name, df_bronze[col_name].cast("string"))

# guardar en tabla Delta
df_bronze.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("workspace.bronze.hist_air_pollution")

In [0]:
%sql
select * from workspace.bronze.hist_air_pollution